In [1]:
from pathlib import Path
import mne
import numpy as np
import json
import csv
import tensorflow as tf
import sys

PROJECT_ROOT = Path.cwd().resolve()

# Nach oben gehen, bis das Datenverzeichnis gefunden wird
while not (PROJECT_ROOT / "data" / "raw" / "ds005363-download").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Projektverzeichnis nicht gefunden.")

    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "ds005363-download"

BUNDLE_DIR = PROJECT_ROOT / "notebooks" / "BunDLe-Net"
if str(BUNDLE_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR))

print("Arbeitsverzeichnis:", Path.cwd())
print("Datenverzeichnis:", DATA_ROOT)

def read_tsv(path: Path) -> list[dict[str, str]]:
    """Liest eine TSV-Datei robust gegen ein UTF-8-BOM ein."""

    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file, delimiter="\t"))

def first_matching(directory: Path, pattern: str) -> Path:
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"Keine Datei mit Muster {pattern!r} in {directory}")
    return matches[0]

def subject_directory(data_root: Path, subject: str) -> Path:
    subject_label = subject if subject.startswith("sub-") else f"sub-{subject}"
    directory = data_root / subject_label / "ses-01" / "eeg"
    if not directory.is_dir():
        raise FileNotFoundError(f"EEG-Verzeichnis nicht gefunden: {directory}")
    return directory

def read_data(subject: str, duration: float, data_root: Path, output_root: Path): 
    eeg_dir = subject_directory(data_root, subject)

    vhdr_path = first_matching(eeg_dir, "*_eeg.vhdr")
    json_path = first_matching(eeg_dir, "*_eeg.json")
    channels_path = first_matching(eeg_dir, "*_channels.tsv")
    events_path = first_matching(eeg_dir, "*_events.tsv")

    with json_path.open("r", encoding="utf-8") as file:
        metadata = json.load(file)
    channels = read_tsv(channels_path)
    events = read_tsv(events_path)

    print(f"Lese BrainVision-Aufnahme ein: {vhdr_path}")
    raw = mne.io.read_raw_brainvision(vhdr_path, preload=False, verbose="ERROR")
    eeg_picks = mne.pick_types(raw.info, eeg=True, exclude=[])
    if len(eeg_picks) == 0:
        raise RuntimeError("Die Aufnahme enthält keine als EEG markierten Kanäle.")

    return raw, channels, events



I0000 00:00:1793982674.937908   60012 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793982674.983075   60012 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793982676.444842   60012 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Arbeitsverzeichnis: /home/bened/bachelor-thesis/notebooks/BunDLe-Net/notebooks
Datenverzeichnis: /home/bened/bachelor-thesis/data/raw/ds005363-download


In [2]:
try:
    raw, channels, events = read_data(
        subject="101",
        duration=60.0,
        data_root=DATA_ROOT,
        output_root=PROJECT_ROOT / "results",
    )

except FileNotFoundError as exc:
    raise SystemExit(f"Datei nicht verfügbar: {exc}") from exc
except RuntimeError as exc:
    raise SystemExit(f"Analyse konnte nicht durchgeführt werden: {exc}") from exc


stimulus_to_state = {}

for code in range(1, 17):
    stimulus_to_state[code] = 1  # Gesicht

for code in range(17, 33):
    stimulus_to_state[code] = 2  # Tier

for code in range(33, 49):
    stimulus_to_state[code] = 3  # Ort

for code in range(49, 65):
    stimulus_to_state[code] = 4  # Objekt

for code in range(65, 70):
    stimulus_to_state[code] = 5

def extract_s(trial_type):
    if not trial_type.startswith("Stimulus/S"):
        return None

    number = trial_type.removeprefix("Stimulus/S").strip()
    return int(number)

labels = []

for event in events:
    code = extract_s(event["trial_type"])

    if code in stimulus_to_state:
        label = stimulus_to_state[code]
    elif code == 199 or code in range(101, 109):
        label = 0  
    else:
        continue 

    sample = int(event["sample"])
        
    labels.append((sample, label))

import queue
import threading

def prep_data(X, B, win=15):
    if win < 1 or len(X) <= win:
        raise ValueError("X must contain more samples than the positive window length.")
    if len(X) != len(B):
        raise ValueError("X and B must have the same number of samples.")
    # Read-only views share the original recording, including overlapping pairs.
    windows = np.lib.stride_tricks.sliding_window_view(
        X, window_shape=win + 1, axis=0
    ).transpose(0, 2, 1)
    X_paired = np.lib.stride_tricks.sliding_window_view(
        windows, window_shape=win, axis=1
    ).transpose(0, 1, 3, 2)
    B_1 = B[win:]
    print(f'Prepped data, X shape {X_paired.shape}, B shape {B_1.shape}')
    return X_paired, B_1



def generate_B(start, neuron_traces_block):
    B = np.zeros(neuron_traces_block.shape[1], dtype=int)
    l = len(labels)

    for e in range(l):
        current_sample = max(0, labels[e][0] - start) 
        label = labels[e][1]

        if e + 1 < l:
            next_sample = min(len(B), labels[e + 1][0] - start)
        else:
            next_sample = len(B)

        if current_sample < next_sample:
            B[current_sample:next_sample] = label

    return B




Lese BrainVision-Aufnahme ein: /home/bened/bachelor-thesis/data/raw/ds005363-download/sub-101/ses-01/eeg/sub-101_ses-01_task-objrecog_eeg.vhdr


In [3]:
from functions import _pca_initialisation, tf_batch_prep, _best_of_5_runs, BunDLeTrainer, tqdm, BunDLeNet
import matplotlib.pyplot as plt

batch_size = 1024
win = 150
block_size = 16*batch_size

model = BunDLeNet(latent_dim=3)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
trainer = BunDLeTrainer(model, optimizer)
gamma=0.09 
all_losses = []


def train_block(X_paired, B_1, batch_size=batch_size):
    perm = np.random.permutation(len(X_paired))   # Shuffling the data using a random permutation
    for s in range(0, len(perm), batch_size):

        idx = perm[s:s + batch_size]

        x = tf.convert_to_tensor(X_paired[idx], dtype=tf.float32)
        b = tf.convert_to_tensor(B_1[idx], dtype=tf.int64)

        DCC_loss, behaviour_loss, total_loss = trainer.train_step(x, b, gamma=gamma)

        all_losses.append([float(DCC_loss), float(behaviour_loss), float(total_loss)])

def build_training_batch(eeg_data, labels, epoch_numbers, start_time_points, window_length):
    """
    eeg_data:  Form (Anzahl Epochs, Anzahl Kanäle, Anzahl Zeitpunkte)
    labels:    Form (Anzahl Epochs, Anzahl Zeitpunkte)

    Rückgabe wie bei prep_data:
    window_pairs:  Form (Anzahl Paare, 2, window_length, Anzahl Kanäle)
    target_labels: Form (Anzahl Paare,)
    """
    window_pairs = []
    target_labels = []

    for epoch_number, start_time_point in zip(epoch_numbers, start_time_points):
        one_epoch = eeg_data[epoch_number].T          # Form (Zeitpunkte, Kanäle)

        first_window = one_epoch[start_time_point : start_time_point + window_length]
        second_window = one_epoch[start_time_point + 1 : start_time_point + window_length + 1]

        window_pairs.append(np.stack([first_window, second_window]))

        last_time_point_of_second_window = start_time_point + window_length
        target_labels.append(labels[epoch_number, last_time_point_of_second_window])

    return np.array(window_pairs, dtype=np.float32), np.array(target_labels)

loss_array = np.array(all_losses)

plt.figure()
for i, label in enumerate(
    [
        "$\mathcal{L}_{{Markov}}$",
        "$\mathcal{L}_{{Behavior}}$",
        "Total loss $\mathcal{L}$",
    ]
):
    plt.semilogy(loss_array[:, i], label=label)
plt.legend()
plt.show()

W0000 00:00:1793982678.077847   60012 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1793982678.079975   60012 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1793982678.162151   60012 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8296 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070, pci bus id: 0000:03:00.0, compute capability: 12.0a


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

<Figure size 640x480 with 0 Axes>

In [ ]:


# EEG-Kanäle auswählen
eeg_picks = mne.pick_types(raw.info, eeg=True, exclude=[])

# Messwerte: (Kanäle, Zeitpunkte)
neuron_traces = raw.get_data(picks=eeg_picks)

# Kanalnamen in derselben Reihenfolge wie die Messwerte
channel_names = np.array(
    [raw.ch_names[i] for i in eeg_picks],
    dtype=object
)

# Abtastrate der Aufnahme: bei dir 1000 Hz
fps = raw.info["sfreq"]
state_names = [
    "keine Kategorie",
    "Gesicht",
    "Tier",
    "Ort",
    "Objekt",
    "Catch",
]

stimulus_to_state = {}

for code in range(1, 17):
    stimulus_to_state[code] = 1  # Gesicht

for code in range(17, 33):
    stimulus_to_state[code] = 2  # Tier

for code in range(33, 49):
    stimulus_to_state[code] = 3  # Ort

for code in range(49, 65):
    stimulus_to_state[code] = 4  # Objekt

for code in range(65, 70):
    stimulus_to_state[code] = 5
print("--------------------------------------- STIMULUS TO STATE ARRAY ---------------------------------------")
print(stimulus_to_state)
print("")
print("")

event = events[4] 

sample = int(event["sample"])
messwerte = neuron_traces[:, sample]

print("--------------------------------------- Neuron Traces ---------------------------------------")
print(neuron_traces[:,:])
print("")
print(neuron_traces[[0],:])
print("")
print(neuron_traces.shape)

print("--------------------------------------- EVENT NR. 4 ---------------------------------------")
print(event)
print("")
print("")
print("--------------------------------------- MESSWERTE FÜR SAMPLE VON EVENT NR. 4 ---------------------------------------")
print(messwerte)
print("")
print("")
print("--------------------------------------- FORM VON MESSWERTE ---------------------------------------")
print(messwerte.shape)
print("")
print("")
print("--------------------------------------- FORM VON NEURON TRACES ---------------------------------------")
print(neuron_traces.shape)
print("")
print("")
# print("--------------------------------------- FORM VON B ---------------------------------------")
# B = np.zeros(neuron_traces.shape[1], dtype=int)
# print(B.shape)